# nb01 — Clean the Plan Quality Files (AQFS, Provider Ratios, Grievances, Encounter Completeness)

<hr style="border: 3px solid black;">

**Purpose:** turn the raw extracts from nb00 into tidy CSVs for the scorecard post (scope decision: AQFS as the core, provider ratios and grievances as companions).

**Cleaning plan (from the nb00 profile):**
1. AQFS: split `Reporting Unit` ('Anthem - Alameda') into Plan and Region; `AQFS` percent text to numeric
2. Provider Ratios: parse `Month` (202301) to a date; both ratio columns to numeric
3. Grievance Type: comma formatted counts to numeric; keep the roll up and detail levels
4. Encounter Completeness: repair the shifted headers (real headers sit in row 1); percent text to numeric
5. QA checks throughout; the demographic files (Population/Age/Sex/Ethnicity) are statewide only and stay raw for now

## 1. Load

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path('..') / 'data'
RAW = DATA_DIR / 'raw'
log = json.loads((DATA_DIR / 'extraction_log.json').read_text())
files = {i['resource']: RAW / i['file'] for i in log['files']}
print(list(files))

['HEDIS', 'Population', 'Provider Ratios', 'Sex', 'Age', 'Encounter Completeness', 'Ethnicity', 'Grievance Demographics', 'Grievance Type']


## 2. AQFS (the core table)

Grain target: one row per Plan x Region x Year, with AQFS as a number 0 to 100.

In [2]:
h = pd.read_csv(files['HEDIS'])

# Split 'Anthem - Alameda' into plan abbreviation and region; ' - ' with spaces avoids splitting hyphenated names
parts = h['Reporting Unit'].str.split(' - ', n=1, expand=True)
h['Plan'] = parts[0].str.strip()
h['Region'] = parts[1].str.strip()

# '53.18%' -> 53.18 (suppressed/blank -> NaN)
h['AQFS'] = pd.to_numeric(h['AQFS'].astype(str).str.rstrip('%').str.strip(), errors='coerce')
h = h.rename(columns={'HEDIS Reporting Year': 'Year'})[['Year', 'Plan', 'Region', 'Reporting Unit', 'AQFS']]

print(h.shape)
print('Years:', sorted(h.Year.unique()))
print('Plans:', sorted(h.Plan.unique()))
print('AQFS range:', h.AQFS.min(), 'to', h.AQFS.max(), '| missing:', int(h.AQFS.isna().sum()))

(436, 5)
Years: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Plans: ['AAH', 'Aetna', 'Anthem', 'Blue Shield', 'CCAH', 'CCHP', 'CH&W', 'CHG', 'CalOptima', 'CalViva', 'CenCal', 'GCHP', 'HPSJ', 'HPSM', 'Health Net', 'IEHP', 'Kaiser', 'Kern', 'LA Care', 'Molina', 'Partnership', 'SCFHP', 'SFHP', 'United']
AQFS range: 26.0 to 98.1 | missing: 0


In [3]:
# QA: find L.A. Care's reporting unit(s) and confirm an unbroken series
la = h[h['Reporting Unit'].str.contains('LA|L.A|Care', case=False, na=False)]
print(la['Reporting Unit'].unique())
print()
print(h[h.Plan.isin(la.Plan.unique())].sort_values('Year')[['Year','Reporting Unit','AQFS']].to_string(index=False))

['AAH - Alameda' 'Anthem - Alameda' 'Anthem - Santa Clara'
 'Anthem - Tulare' 'Health Net - Stanislaus' 'Health Net - Tulare'
 'HPSJ - Stanislaus' 'LA Care - Los Angeles' 'SCFHP - Santa Clara']

 Year           Reporting Unit  AQFS
 2016            AAH - Alameda 53.18
 2016      SCFHP - Santa Clara 60.00
 2016    LA Care - Los Angeles 60.91
 2016        HPSJ - Stanislaus 40.91
 2016       HPSJ - San Joaquin 38.64
 2016      Health Net - Tulare 60.91
 2016  Health Net - Stanislaus 49.09
 2016   Health Net - San Diego 51.36
 2016  Health Net - Sacramento 46.36
 2016 Health Net - Los Angeles 61.36
 2016        Health Net - Kern 46.36
 2016          Anthem - Tulare 59.55
 2016 Health Net - San Joaquin 32.27
 2016   Anthem - San Francisco 58.64
 2016     Anthem - Santa Clara 56.82
 2016    Anthem - Contra Costa 45.00
 2016          Anthem - Fresno 44.09
 2016           Anthem - Kings 46.82
 2016          Anthem - Madera 51.36
 2016         Anthem - Alameda 44.55
 2016        Anthem - Region

In [8]:
# QA: units per year (reporting units come and go; know the panel before charting)
print(h.groupby('Year')['Reporting Unit'].nunique())

# Append a 'Statewide Average' pseudo reporting unit so it appears as a third line in one legend
sw = (h.groupby('Year', as_index=False)['AQFS'].mean()
        .assign(Plan='Statewide Average', Region='All California')
        .rename(columns={}))
sw['Reporting Unit'] = 'Statewide Average'
sw['AQFS'] = sw['AQFS'].round(3)
h = pd.concat([h, sw[h.columns]], ignore_index=True)

h.to_csv(DATA_DIR / 'aqfs_clean.csv', index=False)
print('wrote aqfs_clean.csv', h.shape, '(includes 8 Statewide Average rows)')

Year
2016    54
2017    54
2018    54
2019    54
2020    57
2021    57
2022    57
2023    57
Name: Reporting Unit, dtype: int64
wrote aqfs_clean.csv (452, 5) (includes 8 Statewide Average rows)


## 3. Provider Ratios

In [5]:
pr = pd.read_csv(files['Provider Ratios'])
pr['Month'] = pd.to_datetime(pr['Month'].astype(str), format='%Y%m')
for col in ['PCPs per 2,000 Members', 'Physicians per 1,200 Members']:
    pr[col] = pd.to_numeric(pr[col].astype(str).str.replace(',', ''), errors='coerce')
pr = pr.rename(columns={'Plan Parent Reporting Name': 'Plan'})
print(pr.shape, '| plans:', pr.Plan.nunique(), '| months:', pr.Month.nunique())
pr.to_csv(DATA_DIR / 'provider_ratios_clean.csv', index=False)
print('wrote provider_ratios_clean.csv')

(372, 4) | plans: 31 | months: 12
wrote provider_ratios_clean.csv


## 4. Grievance Type

In [6]:
g = pd.read_csv(files['Grievance Type'])
g['Grievances'] = pd.to_numeric(g['Grievances (PACES)'].astype(str).str.replace(',', ''), errors='coerce')
g = g.drop(columns=['Grievances (PACES)'])
print(g.shape)
print(g.groupby('Grievance Type Roll-up')['Grievances'].sum().sort_values(ascending=False))
g.to_csv(DATA_DIR / 'grievance_type_clean.csv', index=False)
print('wrote grievance_type_clean.csv')

(184, 4)
Grievance Type Roll-up
Quality of Service    216256
Access to Care        136439
Coverage               89629
Quality of Care        63208
Compliance             35139
Referral               30729
Transportation         23672
Name: Grievances, dtype: int64
wrote grievance_type_clean.csv


## 5. Encounter Completeness (header repair)

The raw file has its real headers in the first data row; reread with explicit names.

In [7]:
ec = pd.read_csv(files['Encounter Completeness'], skiprows=1,
                 names=['Plan', 'Grading Period', 'Service Category', 'Pct Complete', 'Color Grade'])

# The raw file has TWO header rows; drop the leftover blank row and the repeated header row
ec = ec[ec['Plan'].notna() & ec['Plan'].ne('Plan Parent Reporting Name')].reset_index(drop=True)

ec['Pct Complete'] = pd.to_numeric(ec['Pct Complete'].astype(str).str.rstrip('%'), errors='coerce')
print(ec.shape)
print(ec.head(8).to_string(index=False))
print('Grading periods:', sorted(ec['Grading Period'].dropna().unique()))
print('QA junk rows remaining:', int((ec['Plan'] == 'Plan Parent Reporting Name').sum() + ec['Plan'].isna().sum()))
ec.to_csv(DATA_DIR / 'encounter_completeness_clean.csv', index=False)
print('wrote encounter_completeness_clean.csv')

(835, 5)
Plan Grading Period              Service Category  Pct Complete Color Grade
 AAH        CY 2018                     Inpatient         99.54       Green
 AAH        CY 2018 Outpatient and Emergency Room         98.38       Green
 AAH        CY 2018                  Prescription         97.47       Green
 AAH        CY 2018                  Professional         95.31       Green
 AAH        CY 2019                     Inpatient         96.13       Green
 AAH        CY 2019 Outpatient and Emergency Room         93.38       Green
 AAH        CY 2019                  Prescription         99.51       Green
 AAH        CY 2019                  Professional        102.26       Green
Grading periods: ['CY 2018', 'CY 2019', 'CY 2021', 'CY 2022', 'Jul19-Feb', 'SFY 17-18', 'SFY 18-19', 'SFY 20-21', 'SFY 21-22']
QA junk rows remaining: 0
wrote encounter_completeness_clean.csv


---

**Next:** paste every printed output into the working session. The plan abbreviation list from section 2 decides whether we need a HARMONIZE map (matching post 1's plan names), and the L.A. Care QA cell confirms the series we build the story around.